# 🧬 NonBScanner: High-Efficiency Big Genome Analysis

**Professional Non-B DNA Motif Detection for Large Genomes**

This notebook provides a streamlined workflow for analyzing large genomic sequences with high efficiency:
- ⚡ **Parallel Processing**: Multi-core execution for maximum speed
- 🎯 **Memory Efficient**: Chunked processing for genomes of any size
- 📊 **Comprehensive Results**: Consolidated Excel output with all motif classes
- 📈 **Publication-Quality Visualizations**: 25+ chart types with statistical analysis
- 🔍 **11 Motif Classes**: Complete Non-B DNA detection suite

---

## 📋 Detected Motif Classes

1. **Curved DNA** - A-tract mediated bending
2. **Slipped DNA** - Direct repeats, STRs
3. **Cruciform** - Inverted repeats
4. **R-Loop** - RNA-DNA hybrids (QmRLFS algorithm)
5. **Triplex** - Three-stranded structures
6. **G-Quadruplex** - Four-stranded G-rich structures (7 subclasses)
7. **i-Motif** - C-rich structures
8. **Z-DNA** - Left-handed helix
9. **A-philic DNA** - A-rich structures
10. **Hybrid** - Multi-class overlaps
11. **Clusters** - High-density regions

---

## ⚙️ Performance Features

- **Chunked Parallel Processing**: Automatic for sequences > 10kb
- **Multi-core Utilization**: All CPU cores engaged
- **Progress Tracking**: Real-time analysis feedback
- **Memory Optimization**: Handles genomes 100MB+
- **Throughput**: ~24,674 bp/second on optimized detectors

---

## 📦 Box 1: Setup and Configuration

**Load libraries, configure parameters, and prepare your genome file**

In [ ]:
# ============================================================================
# IMPORTS AND SETUP
# ============================================================================

import os
import sys
import time
import warnings
from pathlib import Path
from datetime import datetime

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Core NonBScanner modules
import nonbscanner as nbs
from parallel_scanner import ChunkedParallelScanner
from utilities import (
    read_fasta_file,
    parse_fasta,
    export_to_excel,
    export_to_csv,
    export_to_bed,
    analyze_class_subclass_detection,
    print_detection_report,
    calculate_motif_statistics,
    get_basic_stats
)
from visualizations import (
    plot_comprehensive_class_analysis,
    plot_comprehensive_subclass_analysis,
    plot_score_statistics_by_class,
    plot_length_statistics_by_class,
    plot_motif_distribution,
    plot_coverage_map,
    plot_genome_landscape_track,
    plot_sliding_window_heat_ribbon
)

# Scientific computing
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Memory monitoring
import psutil

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("✅ Libraries imported successfully!")
print(f"📍 Working Directory: {os.getcwd()}")
print(f"🐍 Python Version: {sys.version.split()[0]}")
print(f"🧬 NonBScanner Version: {nbs.__version__}")

# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

print("\n" + "="*80)
print("⚙️  CONFIGURATION PARAMETERS")
print("="*80)

# Input file path - UPDATE THIS WITH YOUR FASTA FILE
GENOME_FILE = "example_motifs_multiline.fasta"  # Change to your genome file

# Output directory for results
OUTPUT_DIR = "analysis_results"

# Performance settings
CHUNK_SIZE = 50000      # 50kb chunks for optimal performance
OVERLAP_SIZE = 500      # 500bp overlap to catch boundary motifs
USE_PARALLEL = True     # Enable parallel processing

# Visualization settings
PLOT_DPI = 300          # Publication quality (300 DPI)
SAVE_PLOTS = True       # Save plots to files

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n📁 Input File: {GENOME_FILE}")
print(f"📂 Output Directory: {OUTPUT_DIR}")
print(f"⚡ Parallel Processing: {'Enabled' if USE_PARALLEL else 'Disabled'}")
print(f"🧩 Chunk Size: {CHUNK_SIZE:,} bp")
print(f"🔄 Overlap: {OVERLAP_SIZE:,} bp")
print(f"📊 Plot DPI: {PLOT_DPI}")

# ============================================================================
# LOAD GENOME SEQUENCES
# ============================================================================

print("\n" + "="*80)
print("📖 LOADING GENOME SEQUENCES")
print("="*80)

# Check if file exists
if not os.path.exists(GENOME_FILE):
    print(f"\n⚠️  WARNING: File '{GENOME_FILE}' not found!")
    print("\n📝 Please update the GENOME_FILE variable with the correct path.")
    print("\nExample:")
    print('  GENOME_FILE = "/path/to/your/genome.fasta"')
else:
    # Load sequences from FASTA file
    sequences = read_fasta_file(GENOME_FILE)
    
    print(f"\n✅ Loaded {len(sequences)} sequence(s) from {GENOME_FILE}")
    print("\n📊 Sequence Summary:")
    print("-" * 80)
    
    total_length = 0
    for i, (name, seq) in enumerate(sequences.items(), 1):
        seq_len = len(seq)
        total_length += seq_len
        print(f"{i}. {name[:60]:60s} {seq_len:>12,} bp")
    
    print("-" * 80)
    print(f"{'TOTAL':60s} {total_length:>12,} bp")
    print("-" * 80)
    
    # Memory estimate
    mem_estimate_mb = (total_length * 8) / (1024 * 1024)  # Rough estimate
    print(f"\n💾 Estimated Memory Usage: ~{mem_estimate_mb:.1f} MB")
    
    # Current memory usage
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    current_mem_mb = mem_info.rss / (1024 * 1024)
    print(f"📊 Current Process Memory: {current_mem_mb:.1f} MB")
    
    # Performance estimate
    estimated_time = total_length / 24674  # Using optimized throughput
    print(f"\n⏱️  Estimated Analysis Time: ~{estimated_time:.1f} seconds")
    if estimated_time > 60:
        print(f"   (approximately {estimated_time/60:.1f} minutes)")

print("\n✅ Setup Complete! Ready to run analysis.")
print("\n➡️  Proceed to Box 2 to start the analysis.")

## 🚀 Box 2: Run High-Efficiency Analysis

**Execute parallel motif detection with progress tracking**

In [ ]:
# ============================================================================
# HIGH-EFFICIENCY PARALLEL ANALYSIS
# ============================================================================

print("="*80)
print("🚀 STARTING HIGH-EFFICIENCY GENOME ANALYSIS")
print("="*80)

# Initialize results storage
all_results = {}
analysis_stats = {}

# Initialize parallel scanner for large genomes
if USE_PARALLEL:
    scanner = ChunkedParallelScanner(
        chunk_size=CHUNK_SIZE,
        overlap_size=OVERLAP_SIZE
    )
    print(f"\n✅ Initialized ChunkedParallelScanner")
    print(f"   - Chunk Size: {CHUNK_SIZE:,} bp")
    print(f"   - Overlap: {OVERLAP_SIZE:,} bp")
    print(f"   - Workers: {scanner.max_workers}")
else:
    print("\n⚠️  Parallel processing disabled - using sequential mode")

# Process each sequence
print("\n" + "="*80)
print("🔬 ANALYZING SEQUENCES")
print("="*80)

overall_start_time = time.time()

for seq_idx, (seq_name, sequence) in enumerate(sequences.items(), 1):
    print(f"\n{'='*80}")
    print(f"📝 Sequence {seq_idx}/{len(sequences)}: {seq_name}")
    print(f"{'='*80}")
    print(f"Length: {len(sequence):,} bp")
    
    # Start timing for this sequence
    seq_start_time = time.time()
    
    # Memory before analysis
    mem_before = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
    
    # Run analysis with appropriate method
    if USE_PARALLEL and len(sequence) > 10000:
        print(f"\n⚡ Using parallel chunked processing...")
        motifs = scanner.analyze_sequence(sequence, seq_name, use_parallel=True)
    else:
        print(f"\n🔄 Using standard analysis...")
        motifs = nbs.analyze_sequence(sequence, seq_name)
    
    # Calculate timing and throughput
    seq_elapsed_time = time.time() - seq_start_time
    throughput = len(sequence) / seq_elapsed_time if seq_elapsed_time > 0 else 0
    
    # Memory after analysis
    mem_after = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
    mem_used = mem_after - mem_before
    
    # Store results
    all_results[seq_name] = motifs
    analysis_stats[seq_name] = {
        'length': len(sequence),
        'motifs_found': len(motifs),
        'time_seconds': seq_elapsed_time,
        'throughput_bp_s': throughput,
        'memory_mb': mem_used
    }
    
    # Print results
    print(f"\n✅ Analysis Complete!")
    print(f"   - Motifs Found: {len(motifs):,}")
    print(f"   - Time: {seq_elapsed_time:.2f} seconds")
    print(f"   - Throughput: {throughput:,.0f} bp/s")
    print(f"   - Memory Used: {mem_used:.1f} MB")
    
    # Show class breakdown
    if motifs:
        class_counts = {}
        for motif in motifs:
            cls = motif.get('Class', 'Unknown')
            class_counts[cls] = class_counts.get(cls, 0) + 1
        
        print(f"\n   📊 Motif Class Breakdown:")
        for cls, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"      - {cls}: {count:,} motifs")

# Overall statistics
overall_elapsed_time = time.time() - overall_start_time

print("\n" + "="*80)
print("📊 OVERALL ANALYSIS SUMMARY")
print("="*80)

total_motifs = sum(len(motifs) for motifs in all_results.values())
total_bp = sum(stats['length'] for stats in analysis_stats.values())
avg_throughput = total_bp / overall_elapsed_time if overall_elapsed_time > 0 else 0

print(f"\n✅ Analyzed {len(sequences)} sequence(s)")
print(f"📏 Total Length: {total_bp:,} bp")
print(f"🔍 Total Motifs Found: {total_motifs:,}")
print(f"⏱️  Total Time: {overall_elapsed_time:.2f} seconds")
print(f"⚡ Average Throughput: {avg_throughput:,.0f} bp/s")

# Memory summary
current_mem = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
print(f"💾 Current Memory Usage: {current_mem:.1f} MB")

print("\n✅ Analysis Complete! Proceeding to results export and visualization.")
print("\n➡️  Continue to Box 3 for consolidated Excel output and visualizations.")

## 📊 Box 3: Generate Consolidated Results and Visualizations

**Export comprehensive Excel file and create publication-quality visualizations**

In [ ]:
# ============================================================================
# CONSOLIDATED EXCEL EXPORT
# ============================================================================

print("="*80)
print("📊 GENERATING CONSOLIDATED EXCEL OUTPUT")
print("="*80)

# Combine all motifs from all sequences
all_motifs = []
for seq_name, motifs in all_results.items():
    all_motifs.extend(motifs)

print(f"\n📈 Total motifs to export: {len(all_motifs):,}")

# Generate timestamp for output files
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Export to Excel with multiple sheets
excel_filename = os.path.join(OUTPUT_DIR, f"NonBScanner_Results_{timestamp}.xlsx")
print(f"\n📝 Exporting to Excel: {excel_filename}")

try:
    export_to_excel(all_motifs, excel_filename)
    print(f"✅ Excel file created successfully!")
    print(f"   File: {excel_filename}")
    
    # Show sheet information
    class_counts = {}
    for motif in all_motifs:
        cls = motif.get('Class', 'Unknown')
        class_counts[cls] = class_counts.get(cls, 0) + 1
    
    print(f"\n   📑 Excel Sheets Created:")
    print(f"      - Summary (overview statistics)")
    print(f"      - All_Motifs ({len(all_motifs):,} total motifs)")
    for cls, count in sorted(class_counts.items()):
        print(f"      - {cls} ({count:,} motifs)")
        
except Exception as e:
    print(f"❌ Error exporting to Excel: {e}")
    print(f"   Continuing with other exports...")

# Export to CSV (alternative format)
csv_filename = os.path.join(OUTPUT_DIR, f"NonBScanner_Results_{timestamp}.csv")
print(f"\n📝 Exporting to CSV: {csv_filename}")

try:
    export_to_csv(all_motifs, csv_filename)
    print(f"✅ CSV file created successfully!")
except Exception as e:
    print(f"❌ Error exporting to CSV: {e}")

# Export to BED format (for genome browsers)
bed_filename = os.path.join(OUTPUT_DIR, f"NonBScanner_Results_{timestamp}.bed")
print(f"\n📝 Exporting to BED: {bed_filename}")

try:
    export_to_bed(all_motifs, bed_filename)
    print(f"✅ BED file created successfully!")
    print(f"   Compatible with UCSC Genome Browser and IGV")
except Exception as e:
    print(f"❌ Error exporting to BED: {e}")

# ============================================================================
# DETECTION ANALYSIS REPORT
# ============================================================================

print("\n" + "="*80)
print("🔍 CLASS/SUBCLASS DETECTION ANALYSIS")
print("="*80)

# Analyze which classes and subclasses were detected
detection_report = analyze_class_subclass_detection(all_motifs)
report_text = print_detection_report(detection_report)
print(report_text)

# Save detection report to file
report_filename = os.path.join(OUTPUT_DIR, f"Detection_Report_{timestamp}.txt")
with open(report_filename, 'w') as f:
    f.write(report_text)
print(f"\n💾 Detection report saved to: {report_filename}")

# ============================================================================
# PUBLICATION-QUALITY VISUALIZATIONS
# ============================================================================

print("\n" + "="*80)
print("📈 GENERATING PUBLICATION-QUALITY VISUALIZATIONS")
print("="*80)

if len(all_motifs) > 0:
    # Create figures directory
    figures_dir = os.path.join(OUTPUT_DIR, f"figures_{timestamp}")
    os.makedirs(figures_dir, exist_ok=True)
    print(f"\n📁 Figures directory: {figures_dir}")
    
    # 1. Comprehensive Class Analysis
    print("\n1️⃣  Generating comprehensive class analysis...")
    try:
        fig = plot_comprehensive_class_analysis(all_motifs)
        if SAVE_PLOTS and fig:
            filename = os.path.join(figures_dir, "01_comprehensive_class_analysis.png")
            fig.savefig(filename, dpi=PLOT_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f"   ✅ Saved: {filename}")
        else:
            plt.show()
    except Exception as e:
        print(f"   ⚠️  Error: {e}")
    
    # 2. Comprehensive Subclass Analysis
    print("\n2️⃣  Generating comprehensive subclass analysis...")
    try:
        fig = plot_comprehensive_subclass_analysis(all_motifs)
        if SAVE_PLOTS and fig:
            filename = os.path.join(figures_dir, "02_comprehensive_subclass_analysis.png")
            fig.savefig(filename, dpi=PLOT_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f"   ✅ Saved: {filename}")
        else:
            plt.show()
    except Exception as e:
        print(f"   ⚠️  Error: {e}")
    
    # 3. Score Statistics by Class
    print("\n3️⃣  Generating score statistics by class...")
    try:
        fig = plot_score_statistics_by_class(all_motifs)
        if SAVE_PLOTS and fig:
            filename = os.path.join(figures_dir, "03_score_statistics_by_class.png")
            fig.savefig(filename, dpi=PLOT_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f"   ✅ Saved: {filename}")
        else:
            plt.show()
    except Exception as e:
        print(f"   ⚠️  Error: {e}")
    
    # 4. Length Statistics by Class
    print("\n4️⃣  Generating length statistics by class...")
    try:
        fig = plot_length_statistics_by_class(all_motifs)
        if SAVE_PLOTS and fig:
            filename = os.path.join(figures_dir, "04_length_statistics_by_class.png")
            fig.savefig(filename, dpi=PLOT_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f"   ✅ Saved: {filename}")
        else:
            plt.show()
    except Exception as e:
        print(f"   ⚠️  Error: {e}")
    
    # 5. Motif Distribution
    print("\n5️⃣  Generating motif distribution plot...")
    try:
        fig = plot_motif_distribution(all_motifs)
        if SAVE_PLOTS and fig:
            filename = os.path.join(figures_dir, "05_motif_distribution.png")
            fig.savefig(filename, dpi=PLOT_DPI, bbox_inches='tight')
            plt.close(fig)
            print(f"   ✅ Saved: {filename}")
        else:
            plt.show()
    except Exception as e:
        print(f"   ⚠️  Error: {e}")
    
    # 6. Coverage Map (for first sequence)
    if len(sequences) > 0:
        first_seq_name = list(sequences.keys())[0]
        first_seq = sequences[first_seq_name]
        first_seq_motifs = all_results[first_seq_name]
        
        if len(first_seq_motifs) > 0:
            print(f"\n6️⃣  Generating coverage map for '{first_seq_name}'...")
            try:
                fig = plot_coverage_map(first_seq_motifs, len(first_seq))
                if SAVE_PLOTS and fig:
                    filename = os.path.join(figures_dir, "06_coverage_map.png")
                    fig.savefig(filename, dpi=PLOT_DPI, bbox_inches='tight')
                    plt.close(fig)
                    print(f"   ✅ Saved: {filename}")
                else:
                    plt.show()
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
            
            # 7. Genome Landscape Track (for sequences < 50kb)
            if len(first_seq) < 50000:
                print(f"\n7️⃣  Generating genome landscape track...")
                try:
                    fig = plot_genome_landscape_track(first_seq_motifs, len(first_seq))
                    if SAVE_PLOTS and fig:
                        filename = os.path.join(figures_dir, "07_genome_landscape.png")
                        fig.savefig(filename, dpi=PLOT_DPI, bbox_inches='tight')
                        plt.close(fig)
                        print(f"   ✅ Saved: {filename}")
                    else:
                        plt.show()
                except Exception as e:
                    print(f"   ⚠️  Error: {e}")
    
    print("\n✅ All visualizations generated successfully!")
    print(f"\n📁 Results saved in: {OUTPUT_DIR}")
    
else:
    print("\n⚠️  No motifs found - skipping visualizations")

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("✅ ANALYSIS COMPLETE - FINAL SUMMARY")
print("="*80)

print(f"\n📊 Analysis Statistics:")
print(f"   - Sequences Analyzed: {len(sequences)}")
print(f"   - Total Base Pairs: {total_bp:,}")
print(f"   - Total Motifs Found: {len(all_motifs):,}")
print(f"   - Analysis Time: {overall_elapsed_time:.2f} seconds")
print(f"   - Average Throughput: {avg_throughput:,.0f} bp/s")

print(f"\n📁 Output Files:")
print(f"   - Excel: {excel_filename}")
print(f"   - CSV: {csv_filename}")
print(f"   - BED: {bed_filename}")
print(f"   - Report: {report_filename}")
print(f"   - Figures: {figures_dir}/")

print(f"\n💾 Memory Usage:")
final_mem = psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)
print(f"   - Current: {final_mem:.1f} MB")

print(f"\n🎉 Analysis pipeline completed successfully!")
print(f"\n📖 Check the output directory for all results and visualizations.")
print("="*80)